# San Mateo County Analysis

In [1]:
# Import statements
import pandas as pd
import matplotlib.pyplot as plt

# Display all columns
pd.set_option('display.max_columns', None)

# Import Data

In [2]:
from utils import load_ripa_policing_data

# Load in policing data
#policing = load_ripa_policing_data("sanmateo")

In [5]:
# Load in prosecution data
prosecution = pd.read_excel("https://raw.githubusercontent.com/laurenbchu/honors-thesis/main/data/raw/2014-2022_san-mateo-county_charge-data_redacted-xlsx.xlsx")
prosecution['Year'] = prosecution['OffenseDate'].dt.year

# Select only cases from 2021-2023
prosecution = prosecution[prosecution["Year"].between(2021, 2023)].copy()

In [6]:
# Load in census data
from utils import load_census
census = load_census("06081")

# Policing Analysis

# Prosecution Analysis

In [7]:
# Delete rows with invalid race values
valid_races = (
    "Hispanic",
    "White",
    "Black",
    "All Other",
    "Filipino",
    "Other Asian",
    "Pacific Islander",
    "Chinese",
    "Asian Indian",
    "Vietnamese",
    "Samoan",
    "Hawaiian",
    "Unknown",
    "Korean",
    "American Indian",
    "Japanese",
    "Guamanian",
    "Laotian",
    "Cambodian"
)

prosecution = prosecution[prosecution["Race"].isin(valid_races)].copy()

In [8]:
# Relabel races to fit into categories: White, Hispanic/Latino, Black/African American, Asian, Other

RACE_TO_CENSUS = {
    # Direct mappings
    "White": "White",
    "Hispanic": "Hispanic/Latino",
    "Black": "Black/African American",

    # Asian subgroups → Asian
    "Filipino": "Asian",
    "Other Asian": "Asian",
    "Chinese": "Asian",
    "Asian Indian": "Asian",
    "Vietnamese": "Asian",
    "Korean": "Asian",
    "Japanese": "Asian",
    "Cambodian": "Asian",
    "Laotian": "Asian",

    # Pacific Islander & Native → Other
    "Pacific Islander": "Other",
    "Samoan": "Other",
    "Hawaiian": "Other",
    "Guamanian": "Other",
    "American Indian": "Other",

    # Catch-alls
    "All Other": "Other",
    "Unknown": "Other",
}

prosecution["Census Race"] = prosecution["Race"].map(RACE_TO_CENSUS)

In [9]:
# Create column to indicate if convicted or not
CONVICTED_DISPOSITIONS = {
    "Guilty By Jury",
    "Guilty By Court Trial",
    "Guilty Plea",
    "Guilty Plea Agreement",
    "Plead Guilty Plea Agreement",
    "Plead Guilty Nolo Contendere",
    "Pled No Contest",
    "Nolo",
    "Found Guilty Lesser Included Offense Jury",
    "Found Guilty Lesser Included Offense Court",
    "DEJ Unsuccessful / No Contest Plea",
}

case_convicted = (
    prosecution
    .groupby("FileNumber")["ChargeDisposition"]
    .apply(lambda x: x.isin(CONVICTED_DISPOSITIONS).any())
    .astype(int)
)

prosecution = prosecution.merge(
    case_convicted.rename("was_convicted"),
    on="FileNumber",
    how="left",
)

In [10]:
# Create column to indicate if enhancement or not

ENH_TRUE_DISPOSITIONS = {
    "Enhancement Found True Jury",
    "Enhancement Found True Court",
    "Admit Enhancement",
    "Strike enhancement",
}

case_enhanced = (
    prosecution
    .groupby("FileNumber")["EnhDisposition"]
    .apply(lambda x: x.isin(ENH_TRUE_DISPOSITIONS).any())
    .astype(int)
)

prosecution = prosecution.merge(
    case_enhanced.rename("is_enhancement_charge"),
    on="FileNumber",
    how="left"
)

In [12]:
from utils import compute_prosecution_rates

prosecution_rates = compute_prosecution_rates(
    prosecution,
    race_col="Census Race",
    year_col="Year"
)

prosecution_rates

,Census Race,Year,total_cases,conviction_rate,enhancement_rate
0,Asian,2021.0,2061,0.261038,0.041727
1,Asian,2022.0,290,0.027586,0.000000
2,Black/African American,2021.0,4923,0.226691,0.067032
3,Black/African American,2022.0,720,0.076389,0.005556
4,Hispanic/Latino,2021.0,13909,0.284204,0.068876
5,Hispanic/Latino,2022.0,1917,0.086594,0.003652
6,Other,2021.0,2438,0.242412,0.044299
7,Other,2022.0,303,0.052805,0.016502
8,White,2021.0,8384,0.256679,0.062977
9,White,2022.0,1106,0.084991,0.000904
